# Principal Component Analysis of vehicle safety measures

**CSCI 5612 - Module 2, Project Part 2**

Sixteen separate safety measures describe each vehicle. Are they really sixteen
different things, or a handful of underlying patterns wearing sixteen names?

In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 5612

## 1. Data preparation

PCA needs the same input shape as clustering: a purely numeric matrix with no
labels. Standardising matters even more here. PCA finds directions of maximum
variance, and variance depends on units, so an unstandardised variable with a
wide numeric range would capture the first component purely by being measured
on a larger scale. Standardising means the analysis runs on the **correlation**
matrix, and every variable gets an equal vote.

In [2]:
df = pd.read_csv("../data/clean/vehicle_model_year.csv")
FEATURES = [
    "n_complaints", "crash_rate", "fire_rate", "pct_severe",
    "injuries_per_100_complaints", "median_days_to_report", "mean_vehicle_age",
    "n_recalls", "InvestigationCount", "OverallRating",
    "OverallFrontCrashRating", "OverallSideCrashRating", "RolloverRating",
    "RolloverPossibility", "has_fcw", "has_ldw",
]
X = df[FEATURES].copy()
for col in ["n_complaints", "n_recalls", "InvestigationCount"]:
    X[col] = np.log1p(X[col])
    X = X.rename(columns={col: f"log_{col}"})

print("BEFORE standardising:")
display(X.describe().loc[["mean", "std"]].round(2))

Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
print("AFTER standardising:")
Xs.describe().loc[["mean", "std"]].round(2)

BEFORE standardising:


,log_n_complaints,crash_rate,fire_rate,pct_severe,injuries_per_100_complaints,median_days_to_report,mean_vehicle_age,log_n_recalls,log_InvestigationCount,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,has_fcw,has_ldw
mean,5.23,0.07,0.01,0.09,5.31,12.93,2.99,1.47,0.60,4.83,4.56,4.98,4.24,0.13,0.55,0.53
std,1.25,0.06,0.02,0.06,5.31,9.60,1.32,0.82,0.65,0.36,0.51,0.15,0.48,0.04,0.50,0.50


AFTER standardising:


,log_n_complaints,crash_rate,fire_rate,pct_severe,injuries_per_100_complaints,median_days_to_report,mean_vehicle_age,log_n_recalls,log_InvestigationCount,OverallRating,OverallFrontCrashRating,OverallSideCrashRating,RolloverRating,RolloverPossibility,has_fcw,has_ldw
mean,-0.0,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 2. Eigenvalues and eigenvectors

For the correlation matrix, an **eigenvector** is a direction the matrix
stretches without rotating, and its **eigenvalue** is how much stretching
occurs. In data terms the eigenvectors are the candidate axes and the
eigenvalues are the variance captured along each one. Sorting eigenvectors by
eigenvalue produces the principal components in order of importance - that is
the entire mechanism.

Because the input is standardised, each original variable contributes exactly
1 unit of variance, so **an eigenvalue above 1** marks a component carrying
more information than any single original measure did alone.

In [3]:
pca = PCA(random_state=RANDOM_STATE).fit(Xs)
ev, evr = pca.explained_variance_, pca.explained_variance_ratio_
cum = np.cumsum(evr)

pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(ev))],
    "eigenvalue": ev.round(4),
    "variance_explained_%": (evr * 100).round(2),
    "cumulative_%": (cum * 100).round(2),
})

,component,eigenvalue,variance_explained_%,cumulative_%
0,PC1,3.8846,24.17,24.17
1,PC2,2.3806,14.81,38.99
2,PC3,2.0670,12.86,51.85
3,PC4,1.7985,11.19,63.04
4,PC5,1.2294,7.65,70.69
5,PC6,1.0578,6.58,77.27
6,PC7,0.8106,5.04,82.32
7,PC8,0.7439,4.63,86.95
8,PC9,0.5257,3.27,90.22
9,PC10,0.4220,2.63,92.84


In [4]:
print("components with eigenvalue > 1 (Kaiser):", int((ev > 1).sum()))
print("components needed for 90% variance:    ", int(np.searchsorted(cum, 0.90) + 1))
print("components needed for 95% variance:    ", int(np.searchsorted(cum, 0.95) + 1))

components with eigenvalue > 1 (Kaiser): 6
components needed for 90% variance:     9
components needed for 95% variance:     12


## 3. What the components mean

The loadings say how much each original variable contributes to each component.

In [5]:
loadings = pd.DataFrame(pca.components_[:4].T, index=Xs.columns,
                        columns=["PC1", "PC2", "PC3", "PC4"])
loadings.round(3).sort_values("PC1")

,PC1,PC2,PC3,PC4
log_n_complaints,-0.359,-0.027,-0.165,0.325
mean_vehicle_age,-0.328,0.175,-0.280,-0.199
log_n_recalls,-0.233,-0.127,-0.084,0.429
RolloverPossibility,-0.178,0.251,0.443,0.235
log_InvestigationCount,-0.175,0.014,-0.388,0.276
OverallFrontCrashRating,-0.092,-0.324,-0.197,0.100
fire_rate,0.045,0.211,-0.130,-0.260
median_days_to_report,0.068,0.050,-0.028,-0.394
OverallSideCrashRating,0.081,-0.083,-0.106,0.240
OverallRating,0.133,-0.327,-0.372,0.016


In [6]:
for pc in ["PC1", "PC2"]:
    s = loadings[pc].sort_values()
    print(f"\n{pc}  (explains {evr[int(pc[-1])-1]*100:.1f}% of variance)")
    print("  most negative:", ", ".join(f"{i} {v:+.2f}" for i, v in s.head(3).items()))
    print("  most positive:", ", ".join(f"{i} {v:+.2f}" for i, v in s.tail(3).items()))


PC1  (explains 24.2% of variance)
  most negative: log_n_complaints -0.36, mean_vehicle_age -0.33, log_n_recalls -0.23
  most positive: has_fcw +0.34, pct_severe +0.36, crash_rate +0.37

PC2  (explains 14.8% of variance)
  most negative: OverallRating -0.33, OverallFrontCrashRating -0.32, RolloverRating -0.32
  most positive: crash_rate +0.30, injuries_per_100_complaints +0.34, pct_severe +0.35


## 4. What this says about the topic

**This data does not reduce well.** Nine of sixteen components are needed to
retain 90% of the variance, and the leading component carries under a quarter.
A dataset with strong underlying structure usually puts 60–80% into its first
two or three. So the sixteen measures are close to sixteen genuinely separate
pieces of information, not a few latent factors in disguise — which helps
explain why one star rating leaves so much unsaid.

**PC2 is the more pointed finding.** It loads crash-test ratings against
realised harm with opposite signs, meaning the laboratory measurement and the
ownership record do not merely differ — they vary in opposition along the
strongest axis on which both appear.

Full figures and discussion: <https://rajeshwar987.github.io/csci5612-project/pca.html>